# 05 · Interactive demo: "what could this place look like after a disaster?"

1. Search for a place or pan the map, then draw a **rectangle** (≤ 25 km², inside Kerala).
2. Choose the model (pix2pix or diffusion), the hazard and a rainfall scenario, then press
   **Generate**.
3. Press ▶ to crossfade from today's image (severity 0) to increasingly severe outcomes.

Generated images are plausible scenarios for hazard communication, **not predictions**.
Severities above the range seen in training are marked as extrapolation.
Runtime: GPU recommended (CPU works, slower). Set `GEE_PROJECT` first.

In [ ]:
from google.colab import drive, output

GEE_PROJECT = "your-gee-project-id"  # <- change this
REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

output.enable_custom_widget_manager()  # needed for the map widget in Colab
drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q ".[gee]"

In [ ]:
import io
import itertools
from pathlib import Path

import ee
import geemap
import ipywidgets as widgets
import numpy as np
import torch
from PIL import Image

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data.dataset import load_shared_stats
from kerala_hazard_vision_ai.demo import (
    crossfade,
    fetch_area,
    generate_sweep,
    is_extrapolation,
    rain_plane_cond,
)
from kerala_hazard_vision_ai.evaluate import (
    diffusion_predict,
    gan_predict,
    plot_severity_sweep,
    rgb_uint8,
)
from kerala_hazard_vision_ai.models import load_generator
from kerala_hazard_vision_ai.train_diffusion import load_diffusion

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
FIG_DIR = DRIVE_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GAN_DIR, DIFFUSION_DIR = DRIVE_ROOT / "checkpoints", DRIVE_ROOT / "checkpoints_diffusion"
stats = load_shared_stats(GAN_DIR, DIFFUSION_DIR)
PREDICTORS = {
    "pix2pix": gan_predict(load_generator(GAN_DIR / "best_g.pt", cfg.model, device), device),
    "diffusion": diffusion_predict(
        load_diffusion(DIFFUSION_DIR / "best_ema.pt", cfg, device),
        device, cfg.diffusion.sample_steps, cfg.train.seed,
    ),
}
TRAIN_RAIN_MEAN = float(stats.mean[5])  # channel order: 4 bands, slope, rain, soil
CROSSFADE_STEPS = 30
HAZARD_LABELS = {1: "Landslide", 0: "Flood"}


def tile_hazard(path: Path) -> int:
    with np.load(path) as tile:
        return int(tile["hazard"])


# Offer only hazards the model was trained on (floods may have failed the go/no-go check).
TRAINED_HAZARDS = {tile_hazard(p) for p in (DRIVE_ROOT / "tiles").glob("*.npz")}

In [ ]:
def run_demo(bbox, hazard: int, rain_mm: float, model: str = "pix2pix"):
    """Return (crossfade frames, their severities, today's RGB image, one frame per sweep step)."""
    before, slope, soil = fetch_area(bbox, cfg)
    cond = rain_plane_cond(slope, soil, rain_mm)
    key_frames = generate_sweep(PREDICTORS[model], before, cond, hazard, cfg.demo.sweep, stats,
                                cfg.eval.rgb_max, cfg.data.tile_size)
    today = rgb_uint8(torch.from_numpy(before), cfg.eval.rgb_max).permute(1, 2, 0).numpy()
    key_severities = [0.0, *cfg.demo.sweep]
    frames = crossfade([today, *key_frames], CROSSFADE_STEPS)
    positions = np.linspace(0, len(key_severities) - 1, CROSSFADE_STEPS)
    severities = np.interp(positions, range(len(key_severities)), key_severities)
    return frames, severities, today, key_frames


def to_png(frame: np.ndarray) -> bytes:
    buffer = io.BytesIO()
    Image.fromarray(frame).save(buffer, format="PNG")
    return buffer.getvalue()


m = geemap.Map(center=(10.5, 76.5), zoom=8)
model = widgets.ToggleButtons(options=[("pix2pix (fast)", "pix2pix"), ("Diffusion", "diffusion")])
hazard = widgets.ToggleButtons(
    options=[(HAZARD_LABELS[h], h) for h in sorted(TRAINED_HAZARDS, reverse=True)]
)
rain = widgets.FloatSlider(value=TRAIN_RAIN_MEAN, min=0, max=800, step=10,
                           description="Rain (mm/7d)", continuous_update=False)
generate = widgets.Button(description="Generate", button_style="primary")
status = widgets.HTML()
picture = widgets.Image(format="png", width=400)
play = widgets.Play(min=0, max=CROSSFADE_STEPS - 1, interval=120)
scrub = widgets.IntSlider(min=0, max=CROSSFADE_STEPS - 1, description="Frame")
widgets.jslink((play, "value"), (scrub, "value"))
state = {"frames": [], "severities": []}


def show(_=None):
    if not state["frames"]:
        return
    i = scrub.value
    picture.value = to_png(state["frames"][i])
    severity = state["severities"][i]
    warning = " ⚠️ <b>extrapolation</b>" if is_extrapolation(severity, cfg.demo) else ""
    status.value = f"Severity {severity:.2f}{warning}"


def on_generate(_):
    if m.user_roi is None:
        status.value = "Draw a rectangle on the map first."
        return
    lons, lats = zip(*m.user_roi.bounds().coordinates().getInfo()[0], strict=True)
    bbox = (min(lons), min(lats), max(lons), max(lats))
    status.value = "Fetching satellite data and generating…"
    try:
        frames, severities, *_ = run_demo(bbox, hazard.value, rain.value, model.value)
        state.update(frames=frames, severities=severities)
    except Exception as error:  # show every failure (bad area, Earth Engine, network) in the UI
        status.value = f"⚠️ {type(error).__name__}: {error}"
        return
    scrub.value = 0
    show()


generate.on_click(on_generate)
scrub.observe(show, names="value")
widgets.VBox([m, model, widgets.HBox([hazard, rain, generate]), widgets.HBox([play, scrub]),
              status, picture])

## Saved examples (static, so they show in the committed notebook)

In [ ]:
EXAMPLES = {
    "wayanad": (76.13, 11.46, 76.15, 11.48),
    "pettimudi": (77.00, 10.16, 77.02, 10.18),
}
for (name, bbox), model_name in itertools.product(EXAMPLES.items(), PREDICTORS):
    _, _, today, key_frames = run_demo(bbox, 1, TRAIN_RAIN_MEAN, model_name)
    as_tensor = [torch.from_numpy(f).permute(2, 0, 1) for f in key_frames]
    display(plot_severity_sweep(torch.from_numpy(today).permute(2, 0, 1), as_tensor,
                                cfg.demo.sweep, FIG_DIR / f"demo_{name}_{model_name}.png"))